# 🚀 SUDEPI — GOLDEN HYBRID A100-80GB TRAINING PIPELINE

Notebook ini dikonfigurasi khusus untuk **Dataset Hibrida SUDEPI (~59.000 citra)** menggunakan **NVIDIA A100-SXM4-80GB HBM2e** dan **167.1 GB RAM Sistem** Google Colab Pro.

> **Setelan Presisi Tinggi (Memulihkan Uang Tunggal + Multi-Lembar Tajam + Proteksi Koin):**
> - `batch=64`: ~850 langkah gradien/epoch (~85.000 langkah total), konvergensi matang dan stabil.
> - `mosaic=0.3`: 70% citra tetap uang tunggal resolusi utuh (mencegah single bill miss), 30% mozaik untuk multi-uang.
> - `copy_paste=0.15`: Menempelkan koin di sekitar uang kertas secara realistis.
> - `close_mosaic=15`: Mematikan mozaik di 15 epoch terakhir agar batas bounding box mengunci tajam.
> - `cache='ram'`: 100% citra dimuat ke RAM 167 GB (0% disk I/O latency).
> - `amp=True` + `allow_tf32=True`: Kecepatan maksimal Tensor Cores Ampere A100.
> - Estimasi Durasi: **~3.5 - 4.5 MENIT** untuk 100 epoch!
>
> **Kontrak Sistem Terkunci Rapat (ADR-0001, ADR-0002, ADR-0007, ADR-0009):**
> - **8 Kelas Resmi**: `0: rp1000` s.d. `7: koin` (urutan persis sesuai `TABEL_DENOMINASI` di `src/contracts/uang.ts`).
> - **`imgsz = 320`**: Resolusi hemat komputasi CPU HP (Samsung Galaxy M32).
> - **`nms = False`**: NMS class-agnostic sudah ditangani Farrel di TypeScript.
> - **Bentuk Tensor**: Output ONNX wajib tepat `[1, 12, 2100]` (12 kanal = 4 bbox + 8 kelas, 2100 jangkar).

### 1. Pasang Dependensi & Inisialisasi Monster GPU (A100-80GB TF32)

In [ ]:
!nvidia-smi
!pip install -q ultralytics onnx onnxruntime pyyaml

import torch
print(f"CUDA Tersedia: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🔥 Akselerator Aktif : {gpu_name}")
    print(f"💾 Kapasitas VRAM    : {vram_gb:.1f} GB")
    
    # AKTIFKAN PERFORMA PUNCAK A100 TENSOR CORES
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False
    print("🚀 TensorFloat-32 (TF32) + cuDNN Autotuner Aktif! Monster Mode ON!")

### 2. Ekstrak Dataset Hibrida SUDEPI (8 Kelas, ~59.000 Citra)

**Cara pakai:**
1. Klik ikon folder di sidebar kiri Colab.
2. Tarik (*drag and drop*) berkas `dataset_hibrida_sudepi.zip` dari komputermu ke sidebar Colab.
3. Jalankan sel di bawah ini.

In [ ]:
import os, zipfile, re, shutil, glob

# 1. Cari kandidat zip spesifik
zip_candidates = [
    'dataset_hibrida_sudepi.zip', 'dataset_racikan_sudepi.zip', 'dataset_delta_sudepi.zip', 'dataset_sudepi.zip',
    '/content/dataset_hibrida_sudepi.zip', '/content/dataset_racikan_sudepi.zip', '/content/dataset_delta_sudepi.zip',
    '/content/drive/MyDrive/dataset_hibrida_sudepi.zip', '/content/drive/MyDrive/dataset_racikan_sudepi.zip'
]
zip_path = next((p for p in zip_candidates if os.path.exists(p)), None)

# 2. Fallback deteksi dinamis jika nama berkas sedikit berbeda
if not zip_path:
    dynamic_zips = [f for f in glob.glob('*.zip') if 'sample' not in f] + \
                   [f for f in glob.glob('/content/*.zip') if 'sample' not in f] + \
                   [f for f in glob.glob('/content/drive/MyDrive/*.zip') if 'sample' not in f]
    if dynamic_zips:
        zip_path = dynamic_zips[0]

if not zip_path:
    !ls -lh /content
    raise FileNotFoundError("⚠️ Berkas zip dataset belum ditemukan! Pastikan upload selesai atau izin Drive sudah diberikan.")

print(f"📦 Ditemukan: {zip_path} ({os.path.getsize(zip_path)/(1024*1024):.1f} MB)")
print(f"Mengekstrak {zip_path}...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    # Jika isi zip sudah berawalan 'dataset/', ekstrak langsung ke '.' agar menjadi /content/dataset/
    sample = zip_ref.namelist()[0]
    if sample.startswith('dataset/'):
        zip_ref.extractall('.')
    else:
        os.makedirs('dataset', exist_ok=True)
        zip_ref.extractall('dataset')

# Jika sebelumnya sudah terlanjur terbentuk 'dataset/dataset/', ratakan ke 'dataset/'
if os.path.exists('dataset/dataset/data.yaml'):
    print("Merapikan struktur bersarang dataset/dataset...")
    for item in os.listdir('dataset/dataset'):
        src = os.path.join('dataset/dataset', item)
        dst = os.path.join('dataset', item)
        if os.path.exists(dst):
            if os.path.isdir(dst): shutil.rmtree(dst)
            else: os.remove(dst)
        shutil.move(src, dst)
    shutil.rmtree('dataset/dataset')

# Sesuaikan path di data.yaml agar menunjuk ke direktori absolut /content/dataset
yaml_path = 'dataset/data.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f:
    content = f.read()

content = re.sub(r'path:.*', 'path: /content/dataset', content)
with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(content)

print("\n✅ Ekstraksi selesai dan path: /content/dataset siap!")
with open(yaml_path, 'r') as f:
    print(f.read())

# 🚀 SUDEPI — GOLDEN HYBRID A100-80GB TRAINING PIPELINE

Notebook ini dikonfigurasi khusus untuk **Dataset Hibrida SUDEPI (~59.000 citra)** menggunakan **NVIDIA A100-SXM4-80GB HBM2e** dan **167.1 GB RAM Sistem** Google Colab Pro.

> **Setelan Presisi Tinggi (Memulihkan Uang Tunggal + Multi-Lembar Tajam + Proteksi Koin):**
> - `batch=64`: ~850 langkah gradien/epoch (~85.000 langkah total), konvergensi matang dan stabil.
> - `mosaic=0.3`: 70% citra tetap uang tunggal resolusi utuh (mencegah single bill miss), 30% mozaik untuk multi-uang.
> - `copy_paste=0.15`: Menempelkan koin di sekitar uang kertas secara realistis.
> - `close_mosaic=15`: Mematikan mozaik di 15 epoch terakhir agar batas bounding box mengunci tajam.
> - `cache='ram'`: 100% citra dimuat ke RAM 167 GB (0% disk I/O latency).
> - `amp=True` + `allow_tf32=True`: Kecepatan maksimal Tensor Cores Ampere A100.
> - Estimasi Durasi: **~3.5 - 4.5 MENIT** untuk 100 epoch!
>
> **Kontrak Sistem Terkunci Rapat (ADR-0001, ADR-0002, ADR-0007, ADR-0009):**
> - **8 Kelas Resmi**: `0: rp1000` s.d. `7: koin` (urutan persis sesuai `TABEL_DENOMINASI` di `src/contracts/uang.ts`).
> - **`imgsz = 320`**: Resolusi hemat komputasi CPU HP (Samsung Galaxy M32).
> - **`nms = False`**: NMS class-agnostic sudah ditangani Farrel di TypeScript.
> - **Bentuk Tensor**: Output ONNX wajib tepat `[1, 12, 2100]` (12 kanal = 4 bbox + 8 kelas, 2100 jangkar).

In [ ]:
from ultralytics import YOLO

# Muat bobot dasar resmi YOLOv8-Nano
model = YOLO('yolov8n.pt')

# Mulai pelatihan Dataset Hibrida di A100
results = model.train(
    data='dataset/data.yaml',
    imgsz=320,          # WAJIB 320 (ADR-0001, ADR-0009)
    epochs=100,         # 100 epoch untuk akurasi konvergen optimal
    batch=64,           # OPTIMAL: ~850 langkah/epoch (~85.000 langkah konvergensi matang)
    workers=12,         # Maksimal vCPU server Colab High-RAM
    cache='ram',        # KUNCI UTAMA: Caching 100% citra di RAM 167 GB (0% disk latency)
    amp=True,           # Automatic Mixed Precision (FP16 Tensor Cores)
    plots=False,        # Nol disk bottleneck (hilangkan lag matplotlib plotting per-epoch)
    save_period=-1,     # Hanya simpan bobot terbaik & terakhir
    patience=30,        # Early stopping otomatis jika mAP sudah puncak
    # --- Presisi Tinggi: Uang Tunggal Tajam + Multi-Lembar + Koin ---
    degrees=15.0,       # Uang dipegang miring
    shear=5.0,
    perspective=0.0005, # Sudut pandang kamera ponsel
    hsv_v=0.5,          # Simulasi lapak pasar temaram/redup
    hsv_s=0.7,
    fliplr=0.5,
    mosaic=0.3,         # 70% uang tunggal alami ukuran penuh, 30% multi-uang mozaik
    close_mosaic=15,    # Matikan mosaic di 15 epoch terakhir agar bounding box presisi tajam
    mixup=0.0,          # Nol transparan bayangan agar uang tunggal tidak miss
    copy_paste=0.15     # Tempelan koin di atas uang kertas (proteksi koin maksimal)
)

print("🎉 Pelatihan Dataset Hibrida di A100 selesai dalam ~4 menit!")

### 4. Ekspor ke Format ONNX (Wajib `nms=False`)
NMS ditangani oleh decoder TypeScript Farrel di HP, jadi ekspor WAJIB mematikan NMS bawaan.

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import shutil

# Cari bobot terbaik hasil training
save_dir = Path(results.save_dir) if hasattr(results, 'save_dir') else Path('runs/detect/train')
best_pt = save_dir / 'weights' / 'best.pt'
print(f"Mengekspor dari: {best_pt}")

model_best = YOLO(str(best_pt))
exported_path = model_best.export(
    format='onnx',
    imgsz=320,
    opset=12,
    simplify=True,
    nms=False,          # WAJIB False (ADR-0002)
    dynamic=False
)

# Salin ke nama resmi sudepi.onnx
shutil.copy2(exported_path, 'sudepi.onnx')
print("✅ sudepi.onnx berhasil dibuat dan siap diverifikasi!")

### 5. Verifikasi Keselarasan Tensor Output
Output WAJIB `[1, 12, 2100]` (12 kanal = 4 koordinat + 8 kelas, 2100 jangkar).

In [ ]:
import onnxruntime as ort
import numpy as np

session = ort.InferenceSession('sudepi.onnx')
inp_node = session.get_inputs()[0]
out_node = session.get_outputs()[0]

print(f"Node Masukan : nama='{inp_node.name}', bentuk={inp_node.shape}")
print(f"Node Keluaran: nama='{out_node.name}', bentuk={out_node.shape}")

shape = out_node.shape
if shape[1] == 12 and shape[2] == 2100:
    print("\n=======================================================")
    print("🎉 VERIFIKASI SUKSES: Model 100% cocok dengan kode Farrel!")
    print("=======================================================")
else:
    print(f"\n❌ GALAT FATAL: Bentuk output {shape} tidak cocok dengan [1, 12, 2100]!")

### 6. (Opsional) Uji Gerbang Mutu Kuantisasi INT8
Sesuai aturan Farrel di `model/ekspor.py`:
> Ukur mAP@0.5 model INT8 terhadap FP32 pada data validasi.
> Jika mAP turun > 3.0 poin, INT8 DITOLAK dan kita tetap memakai FP32.

In [ ]:
try:
    from onnxruntime.quantization import QuantType, quantize_dynamic
    
    print("Mengukur mAP FP32...")
    val_fp32 = model_best.val(data='dataset/data.yaml', imgsz=320, verbose=False)
    map_fp32 = float(val_fp32.box.map50) * 100
    
    print("Membuat sudepi-int8.onnx...")
    quantize_dynamic('sudepi.onnx', 'sudepi-int8.onnx', weight_type=QuantType.QUInt8)
    
    print("Mengukur mAP INT8...")
    val_int8 = YOLO('sudepi-int8.onnx').val(data='dataset/data.yaml', imgsz=320, verbose=False)
    map_int8 = float(val_int8.box.map50) * 100
    
    drop = map_fp32 - map_int8
    print(f"\nFP32 mAP@0.5: {map_fp32:.2f}%")
    print(f"INT8 mAP@0.5: {map_int8:.2f}% (Turun {drop:.2f} poin)")
    
    if drop <= 3.0:
        print("✅ INT8 DITERIMA! Mengganti sudepi.onnx dengan versi INT8.")
        shutil.copy2('sudepi-int8.onnx', 'sudepi.onnx')
    else:
        print("⚠️ INT8 DITOLAK: Penurunan mAP > 3 poin. Tetap menggunakan FP32.")
except Exception as e:
    print(f"Kuantisasi INT8 dilewati/galat: {e}")

### 7. Unduh Model untuk Farrel
Unduh berkas `sudepi.onnx` dan letakkan di `F:\Code\public\model\sudepi.onnx`.

In [ ]:
from google.colab import files
files.download('sudepi.onnx')